# Analiza dostępności i usług sklepów Żabka w Polsce

Autorzy:
- Jakub Rosiak 251620,
- Mateusz Kosowski 251558,
- Nikodem Nowak 251598

Nasz projekt koncentruje się na analizie rozmieszczenia sieci sklepów Żabka w Polsce, wykorzystując zbiór danych o lokalizacji prawie 10 tysięcy placówek (stan na rok 2024) oraz dane demograficzne z Narodowego Spisu Powszechnego 2021 (siatka kilometrowa GUS). Poprzez integrację danych punktowych z warstwą demograficzną, projekt pozwala na wyznaczenie obszarów o wysokim potencjale inwestycyjnym. Rezultatem prac jest zestaw interaktywnych wizualizacji oraz wniosków biznesowych wspierających procesy decyzyjne w zakresie ekspansji sieci.

### Cele projektu:
- <b>Prezentacja danych statystycznych sieci:</b>
    - Analiza struktury usług dodatkowych
    - Ranking województw i miast pod względem liczby placówek.
- <b>Analiza przestrzenna i demograficzna:</b>
    - Wizualizacja rozmieszczenia sklepów na mapie Polski
    - Wizualizacja liczby mieszkańców przypadających na jeden sklep w siatce kilometrowej.
    - Oszacowanie liczby Polaków posiadających sklep w bezpośrednim sąsiedztwie (analiza dostępności w oparciu o siatkę).
- <b>Wnioski biznesowe:</b>
    - Wskazanie obszarów o najwyższym potencjale inwestystycyjnym




In [ ]:
# Importy
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster
import warnings
from branca.element import MacroElement
from jinja2 import Template


# Config
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
warnings.filterwarnings('ignore')

In [ ]:
# Wczytywanie danych
df_zabka_shop = pd.read_csv("data/zabka_shops.csv", sep=",")
df_clean_zabka_shop = df_zabka_shop[
    (df_zabka_shop['lat'] > 49) & (df_zabka_shop['lat'] < 55) &
    (df_zabka_shop['lng'] > 14) & (df_zabka_shop['lng'] < 25)
].copy()
df_clean_zabka_shop['services'] = df_clean_zabka_shop['services'].fillna('').astype(str)
print(f"Liczba sklepów Żabka w Polsce (2024r.): {len(df_clean_zabka_shop)}")

try:
    gdf_population = gpd.read_file("data/GRID_NSP2021_RES/GRID_NSP2021_RES.shp")
    gdf_population.set_crs(epsg=2180, allow_override=True, inplace=True)
    print(f"Siatka wczytana. Układ: {gdf_population.crs}")
except Exception as e:
     print(f"Błąd wczytywania siatki: {e}")

# Przygotowanie GeoDataFrame w układzie 4326 dla Folium
gdf_zabka_shops_4326 = gpd.GeoDataFrame(
    df_clean_zabka_shop,
    geometry=gpd.points_from_xy(df_clean_zabka_shop.lng, df_clean_zabka_shop.lat),
    crs="EPSG:4326"
)

# Przygotowanie GeoDataFrame w układzie 2180 do łączenia z siatką GUS
if 'gdf_population' in locals():
    gdf_zabka_shops_2180 = gdf_zabka_shops_4326.to_crs(epsg=2180)


# Prezentacja danych statystycznych sieci

In [ ]:
# Przygotowanie
services_exploded = df_clean_zabka_shop['services'].str.split(',').explode().str.strip()
services_counts = services_exploded.value_counts()
legend_map = {
    'ZBC': 'Żabka Café (Kawa/HotDog)', 'ODP': 'Odpiek Pieczywa',
    'PAC': 'Paczki (Odbiór/Nadanie)', 'TER': 'Płatność Kartą',
    'GSM': 'Doładowania Telefonu', 'KPO': 'Karty Podarunkowe',
    'RAC': 'Opłacanie Rachunków', 'REJ': 'Rejestracja SIM',
    'DEN': 'Usługi Energetyczne', 'LOT': 'Lotto',
    'BIH': 'Cashback (Wypłata)', 'DKM': 'Karta Miejska'
}
services_names = [legend_map.get(code, code) for code in services_counts.index]

# Funkcja pomocnicza do etykiet
def add_label(ax):
      for p in ax.patches:
        width = p.get_width()
        if width > 0:
            ax.annotate(f'{int(width)}',
                        (width, p.get_y() + p.get_height() / 2),
                        ha = 'left', va = 'center',
                        xytext = (5, 0), textcoords = 'offset points',
                        fontsize=10, color='black')

# Miasta
top_cities = df_clean_zabka_shop['city'].value_counts().head(10)

# Województwa
voivodeships = df_clean_zabka_shop['voivodeship'].value_counts()

# Generowanie wykresów
fig, axes = plt.subplots(3, 1, figsize=(12, 18))

# Wykres 1
sns.barplot(x=services_counts.values, y=services_names, palette="viridis", ax=axes[0])
axes[0].set_title("Struktura usług dodatkowych", fontsize=16, fontweight='bold', color='black')
axes[0].set_xlabel("Liczba placówek", fontsize=12, color='black')
axes[0].set_ylabel("Rodzaj usługi", fontsize=12, color='black')
add_label(axes[0])

# Wykres 2
sns.barplot(x=voivodeships.values, y=voivodeships.index, palette="mako", ax=axes[1])
axes[1].set_title("Liczba sklepów wg województw", fontsize=16, fontweight='bold', color='black')
axes[1].set_xlabel("Liczba sklepów", fontsize=12, color='black')
axes[1].set_ylabel("Województwo", fontsize=12, color='black')
add_label(axes[1])

# Wykres 3
sns.barplot(x=top_cities.values, y=top_cities.index, palette="rocket", ax=axes[2])
axes[2].set_title("Top 10 miast z największą liczbą Żabek", fontsize=16, fontweight='bold', color='black')
axes[2].set_xlabel("Liczba sklepów", fontsize=12, color='black')
axes[2].set_ylabel("Miasto", fontsize=12, color='black')
add_label(axes[2])

plt.tight_layout()
plt.show()


# Analiza przestrzenna i demograficzna

In [ ]:
# Mapa Polski
map_of_poland = folium.Map(location=[52.0, 19.0], zoom_start=6, tiles=None)

folium.TileLayer(
    tiles="CartoDB dark_matter",
    name="Tło mapy (Ciemne)",
    control=True
).add_to(map_of_poland)

folium.TileLayer(
    tiles="CartoDB positron",
    name="Tło mapy (Jasne)",
    control=True
).add_to(map_of_poland)

# Dodanie warstwy z żabkami
layer_name_html = "<span style='color: #008000; font-weight: bold; font-size: 14px;'>🐸 Sklepy Żabka (Klastry)</span>"
zabka_shops_layer = folium.FeatureGroup(name=layer_name_html, show=True)
marker_cluster = MarkerCluster().add_to(zabka_shops_layer)

# Dodanie adresu i ikony
for row in df_clean_zabka_shop.itertuples():
    folium.Marker(
        location=[row.lat, row.lng],
        tooltip=f"Adres: {row.city}, {row.address}",
        icon=folium.Icon(color="green", icon="frog", prefix="fa")
    ).add_to(marker_cluster)
zabka_shops_layer.add_to(map_of_poland)

In [ ]:
# Spatial Join (Łączenie sklepów z siatką)
joined_data = gpd.sjoin(gdf_zabka_shops_2180, gdf_population, how="inner", predicate="within")

# Ile sklepów w każdym kwadracie - dodanie nowej kolumny do oryginalnej mapy
shops_in_grid = joined_data['index_right'].value_counts()
gdf_population['frog_shop_count'] = 0
gdf_population.loc[shops_in_grid.index, 'frog_shop_count'] = shops_in_grid

# Wyliczenie - bierzemy tylko kwadraty, gdzie jest min. 1 sklep
gdf_analysis = gdf_population[gdf_population['frog_shop_count'] > 0].copy()
gdf_analysis['people_per_shop'] = gdf_analysis['RES'] / gdf_analysis['frog_shop_count']

gdf_analysis_map = gdf_analysis.to_crs(epsg=4326)
max_val = gdf_analysis['people_per_shop'].max()
custom_bins = [0, 2500, 5000, 7500, 10000, 12500, max_val + 1]

# Tworzymy warstwę z choropleth
choropleth = folium.Choropleth(
    geo_data=gdf_analysis_map,
    data=gdf_analysis_map,
    columns=['CODE', 'people_per_shop'],
    key_on='feature.properties.CODE',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0,
    legend_name='Liczba mieszkańców na 1 Żabkę',
    highlight=True,
    name="<span style='color: #d63031; font-weight: bold; font-size: 14px;'>🔴 Obciążenie (Ludzie na sklep)</span>",
    show=False,
    bins= custom_bins
)

tooltip = folium.GeoJsonTooltip(
    fields=['RES', 'frog_shop_count', 'people_per_shop'],
    aliases=['Liczba ludności:', 'Liczba Żabek:', 'Ludzi na sklep:'],
    localize=True,
    sticky=False,
    labels=True,
    style="background-color: white; border: 1px solid black; border-radius: 3px;"
)

choropleth.geojson.add_child(tooltip)
choropleth.add_to(map_of_poland)

In [ ]:
# Nowa kolumna z domyślną wartością daleko
gdf_population['zone_class'] = 'C (Daleko)'

# Dla kwadratów z żabką dajemy tier A
gdf_population.loc[gdf_population['frog_shop_count'] > 0, 'zone_class'] = 'A (W zasięgu)'

# Tworzymy strefe z Buforem. Jest to po to, że jak ktoś nie będzie A, ale będzie w tej strefie to jest B
zone_A_geo = gdf_population[gdf_population['zone_class'] == 'A (W zasięgu)'].geometry
buffer_A = zone_A_geo.buffer(1100).unary_union
mask_B = (gdf_population['zone_class'] == 'C (Daleko)') & \
         (gdf_population['RES'] > 200) & \
         (gdf_population.intersects(buffer_A))
# Musi mieć ponad 200 mieszkańców (pomijamy małe wsie)
gdf_population.loc[mask_B, 'zone_class'] = 'B (Sąsiedztwo)'

# Kopiujemy tylko obszary gdzie ponad 200 osób
gdf_bubbles = gdf_population[gdf_population['RES'] > 200].copy()

gdf_bubbles_4326 = gdf_bubbles.to_crs(epsg=4326)
gdf_bubbles_4326['centroid'] = gdf_bubbles_4326.geometry.centroid
gdf_bubbles_4326 = gdf_bubbles_4326.set_geometry('centroid')
gdf_bubbles_4326['lat'] = gdf_bubbles_4326.geometry.y
gdf_bubbles_4326['lon'] = gdf_bubbles_4326.geometry.x

bubbles_layer = folium.FeatureGroup(
    name="<span style='color: #e67e22; font-weight: bold; font-size: 14px;'>🎈 Dostępność (Bąbelki)</span>",
    show=False
)

def get_bubble_color(zone):
    if 'A' in zone: return '#00ff00' # Zielony (Jest sklep)
    if 'B' in zone: return '#ffff00' # Żółty (Blisko)
    return '#ff0000'                 # Czerwony (Daleko)


for row in gdf_bubbles_4326.itertuples():
    radius = row.RES / 200
    if radius < 5: radius = 5
    if radius > 20: radius = 20

    folium.CircleMarker(
        location=[row.lat, row.lon],
        radius=radius,
        color=get_bubble_color(row.zone_class),
        fill=True,
        fill_color=get_bubble_color(row.zone_class),
        fill_opacity=0.5,
        weight=0,
        tooltip=f"Ludność: {int(row.RES)}<br>Strefa: {row.zone_class}"
    ).add_to(bubbles_layer)
bubbles_layer.add_to(map_of_poland)

In [ ]:
custom_css = """
<style>
    .leaflet-control-layers {
        border-radius: 12px !important;
        box-shadow: 0 4px 12px rgba(0,0,0,0.3) !important;
        padding: 10px !important;
        background-color: rgba(255, 255, 255, 0.95) !important;
        border: 1px solid #ddd !important;
    }
    .leaflet-control-layers label {
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif !important;
        margin-bottom: 5px !important;
        cursor: pointer !important;
    }
</style>
"""

map_of_poland.get_root().html.add_child(folium.Element(custom_css))
folium.LayerControl(collapsed=True).add_to(map_of_poland)

In [ ]:
# Sztuczka aby powiązać legende z wartstwą
class BindColormap(MacroElement):

    def __init__(self, layer, colormap):
        super(BindColormap, self).__init__()
        self.layer = layer
        self.colormap = colormap
        self._template = Template(u"""
        {% macro script(this, kwargs) %}
            var layer = {{this.layer.get_name()}};
            var colormap = {{this.colormap.get_name()}};
            var map = {{this.layer._parent.get_name()}};

            var legend = document.querySelector('.legend');
            if(legend) legend.style.display = 'none';
            map.on('overlayadd', function (eventLayer) {
                if (eventLayer.layer === layer) {
                    var legends = document.getElementsByClassName('legend');
                    for (var i = 0; i < legends.length; i++) {
                        legends[i].style.display = 'block';
                    }
                }
            });

            map.on('overlayremove', function (eventLayer) {
                if (eventLayer.layer === layer) {
                    var legends = document.getElementsByClassName('legend');
                    for (var i = 0; i < legends.length; i++) {
                        legends[i].style.display = 'none';
                    }
                }
            });
        {% endmacro %}
        """)
map_of_poland.add_child(BindColormap(choropleth, choropleth.color_scale))

map_of_poland